# Предсказание пола по транзакциям — Sber 2026 DS

**Цель:** ROC AUC > 0.88 и Accuracy > 0.80

**Решение:** `solution_super_200.py` — super-ensemble: все типы фич (685 кандидатов) → top-200 → 5-fold LGB.

## 1. Установка зависимостей

In [1]:
import sys
import subprocess

sys.path.insert(0, "src")  # modules live in src/

subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt", "-q"], check=True)
print("Dependencies installed.")

Dependencies installed.


## 2. Загрузка данных

Скачивает `train.csv`, `val.csv`, `test.csv` из HuggingFace в `./data/`.

In [2]:
import os
import download_dataset

os.makedirs("data", exist_ok=True)

if all(os.path.exists(f"data/{s}.csv") for s in ["train", "val", "test"]):
    print("Data already downloaded.")
else:
    download_dataset  # module runs on import; re-import triggers re-run
    print("Data downloaded.")

for split in ["train", "val", "test"]:
    size = os.path.getsize(f"data/{split}.csv") // 1024 // 1024
    print(f"  data/{split}.csv  {size} MB")

/home/tony/miniconda3/envs/sber/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Data already downloaded.
  data/train.csv  528 MB
  data/val.csv  77 MB
  data/test.csv  151 MB


## 3. Вычисление gender-embeddings (spaCy)

Считает косинусную близость описаний MCC/TR к мужским/женским словам.  
Сохраняет `results/gender_embeddings/mcc_gender.csv` и `tr_gender.csv`.

In [3]:
import importlib
import pandas as pd
import compute_gender_embeddings

if (os.path.exists("results/gender_embeddings/mcc_gender.csv") and
    os.path.exists("results/gender_embeddings/tr_gender.csv")):
    print("Gender embeddings already computed.")
else:
    import spacy.cli
    spacy.cli.download("ru_core_news_md")

    importlib.reload(compute_gender_embeddings)
    compute_gender_embeddings.main()

mcc_emb = pd.read_csv("results/gender_embeddings/mcc_gender.csv")
tr_emb  = pd.read_csv("results/gender_embeddings/tr_gender.csv")
print(f"MCC embeddings: {len(mcc_emb)} codes")
print(f"TR  embeddings: {len(tr_emb)} types")
display(mcc_emb.head())

Gender embeddings already computed.
MCC embeddings: 184 codes
TR  embeddings: 74 types


,mcc_code,mcc_code_desc,mcc_male_sim,mcc_female_sim,mcc_gender_diff
0,0,"Звонки с использованием телефонов, считывающих...",0.093799,0.033258,0.060542
1,1,Финансовые институты — снятие наличности автом...,-0.157687,-0.175698,0.018012
2,2,Денежные переводы,-0.161267,-0.044517,-0.116750
3,3,"Различные продовольственные магазины — рынки, ...",0.030180,-0.058833,0.089013
4,4,Станции техобслуживания,-0.207843,-0.117434,-0.090409


## 4. Обучение модели

**Super-200:** все типы фич (685 кандидатов) → top-200 по LGB importance → 5-fold ensemble.

In [4]:
import importlib
import solution_super_200
importlib.reload(solution_super_200)

metrics = solution_super_200.main()

Loading splits...
Generating base features (A)...


/home/tony/repos/sber2026ds/src/solution.py:152: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  mcc_df[f'mcc{mcc}_share'] = mcc_df[f'mcc{mcc}_cnt'] / (base['tx_count'] + 1e-6)
/home/tony/repos/sber2026ds/src/solution.py:172: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  feats.reset_index(inplace=True)
/home/tony/repos/sber2026ds/src/solution.py:152: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once

Adding target encoding...
Computing extra features (B)...
Computing MCC interactions + depth (C+D, top-20)...
Computing holiday features (E)...
Computing gender embedding features (F)...
Total candidate features: 685

Step 1: feature selection...
[200]	valid_0's binary_logloss: 0.427735
[400]	valid_0's binary_logloss: 0.428022
  best_iter (×1.1): 326
  Feature groups in top-200:
    A_base: 95
    B_extra: 24
    C_mcc_time: 35
    D_mcc_depth: 12
    E_holiday: 25
    F_gender_emb: 9

Step 2: 5-fold ensemble on top-200...
  fold 1/5  OOF AUC: 0.8642
  fold 2/5  OOF AUC: 0.8775
  fold 3/5  OOF AUC: 0.8632
  fold 4/5  OOF AUC: 0.8731
  fold 5/5  OOF AUC: 0.8678

=== Test metrics ===
  auc_score:       0.8823
  accuracy_score:  0.8071
  precision_score: 0.8356
  recall_score:    0.7232
  train_time:      136.2s

ROC AUC > 0.88: OK  |  Accuracy > 0.80: OK
Saved models → results/solution_2026_04_15_super_200/models.pkl
Saved predictions → results/solution_2026_04_15_super_200/test_predicti

## 5. Итоговые метрики

In [5]:
auc = metrics["auc"]
acc = metrics["accuracy"]

print("=" * 40)
print(f"  Model:     super_200  (200 features)")
print(f"  ROC AUC:   {auc}  {'✓' if auc > 0.88 else '✗'} (target > 0.88)")
print(f"  Accuracy:  {acc}  {'✓' if acc > 0.80 else '✗'} (target > 0.80)")
print(f"  Precision: {metrics['precision']}")
print(f"  Recall:    {metrics['recall']}")
print(f"  Train:     {metrics['train_time']}s")
print("=" * 40)

  Model:     super_200  (200 features)
  ROC AUC:   0.8823  ✓ (target > 0.88)
  Accuracy:  0.8071  ✓ (target > 0.80)
  Precision: 0.8356
  Recall:    0.7232
  Train:     136.2s


## 5.1. Предсказания на тестовой выборке

In [6]:
import pandas as pd

pred_path = "results/solution_2026_04_15_super_200/test_predictions.csv"
df_pred = pd.read_csv(pred_path)
print(f"Total rows: {len(df_pred)}")
display(df_pred[["customer_id", "gender", "prediction", "prediction_proba"]].head(10))

Total rows: 754368


,customer_id,gender,prediction,prediction_proba
0,8,0,0,0.156687
1,8,0,0,0.156687
2,8,0,0,0.156687
3,8,0,0,0.156687
4,8,0,0,0.156687
5,8,0,0,0.156687
6,8,0,0,0.156687
7,8,0,0,0.156687
8,8,0,0,0.156687
9,8,0,0,0.156687


## 6. Сравнение всех экспериментов

In [7]:
import json
import pandas as pd

with open("results/overall.json") as f:
    overall = json.load(f)

rows = []
for e in overall:
    m = e["metrics"]
    rows.append({
        "branch": e["branch"].replace("solution_2026_04_15_", ""),
        "AUC": m["auc_score"],
        "Accuracy": m["accuracy_score"],
        "n_features": e.get("n_features", ""),
        "train_s": e.get("timing", {}).get("train_time_s", ""),
    })

df_res = pd.DataFrame(rows).sort_values("AUC", ascending=False).reset_index(drop=True)
df_res["AUC ✓"] = df_res["AUC"].apply(lambda x: "✓" if x > 0.88 else "")
df_res["Acc ✓"] = df_res["Accuracy"].apply(lambda x: "✓" if x > 0.80 else "")
display(df_res)

,branch,AUC,Accuracy,n_features,train_s,AUC ✓,Acc ✓
0,super_200,0.8823,0.8071,200,136.2,✓,✓
